In [1]:
import pandas as pd
import numpy as np
import os
import time
import re
from huggingface_hub import login, HfApi, hf_hub_download
from kaggle_secrets import UserSecretsClient
from rich import print as rprint

user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
TMDB_API_KEY = user_secrets.get_secret("TMDB_API_KEY")

login(token = HF_TOKEN)

api = HfApi()

In [2]:
catalog_path = hf_hub_download(
    repo_id = "Subhadip007/UERP_Dataset",
    filename = "unified_catalog_v4.parquet",
    repo_type = "dataset",
    token = HF_TOKEN,
)

catalog = pd.read_parquet(catalog_path)

os.makedirs('/kaggle/working/processed', exist_ok = True)

rprint(f"Loaded Catalog: {catalog.shape}")
rprint(catalog.columns.tolist())

unified_catalog_v4.parquet:   0%|          | 0.00/10.3M [00:00<?, ?B/s]

Loaded Catalog: (38984, 14)

[
    'content_id',
    'is_anime',
    'title',
    'content_type',
    'year',
    'genres',
    'overview',
    'rating_normalized',
    'popularity_signal',
    'poster_url',
    'runtime_minutes',
    'episodes',
    'country',
    'language'
]

---

## **Todo List:**
1. **Genre encoding** (multi-hot, 32 canonical genres)
2. **Text Embeddings** (overview &rarr; dense vector, core content-based similarity)
3. **Numerical features** (year, runtime, rating &rarr; clean/normalize)
4. **Popularity normalization** (per-source, which we flagged in _**EDA**_)
5. Combine all and prepare a **final features matrix**

---

### **Step 1: Genre Encoding**

In [3]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

genre_matrix = mlb.fit_transform(catalog['genres'])
genre_df = pd.DataFrame(genre_matrix, columns = [f'genre_{g}' for g in mlb.classes_], index = catalog.index)

rprint(f"Genre matrix shape: {genre_df.shape}")
rprint(f"Genre encoded: {list(mlb.classes_)}")
rprint()
display(genre_df.head(3))

Genre matrix shape: (38984, 32)

Genre encoded: ['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 
'Ecchi', 'Family', 'Fantasy', 'Film-Noir', 'Game-Show', 'History', 'Horror', 'Mahou Shoujo', 'Mecha', 'Music', 
'Musical', 'Mystery', 'News', 'Psychological', 'Reality', 'Romance', 'Science Fiction', 'Slice of Life', 'Sports', 
'Supernatural', 'Talk Show', 'Thriller', 'War', 'Western']

,genre_Action,genre_Adventure,genre_Animation,genre_Biography,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Ecchi,genre_Family,...,genre_Reality,genre_Romance,genre_Science Fiction,genre_Slice of Life,genre_Sports,genre_Supernatural,genre_Talk Show,genre_Thriller,genre_War,genre_Western
0,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,0,1,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,1,0,0


### **Step 2: Text Embedding**

In [4]:
catalog['overview_filled'] = catalog['overview'].fillna('')
missing_mask = catalog['overview_filled'].str.strip() == ''
rprint(f"Missing/empty overview count: {missing_mask.sum()}")

def build_fallback_text(row):
    text = row['overview_filled'].strip()
    if text:
        return text

    genres_str = ', '.join(row['genres']) if len(row['genres']) > 0 else ''
    return f"{row['title']}. Genres: {genres_str}."

catalog['text_for_embedding'] = catalog.apply(build_fallback_text, axis = 1)
rprint(catalog[['title', 'text_for_embedding']].sample(3))

Missing/empty overview count: 105

title                                 text_for_embedding
4585         The Choice  Travis and Gabby first meet as neighbors in a ...
13636  Ailecek Saskiniz  Ferhat is a spoiled man who takes over his fat...
22907    The Restaurant  Monday May 7, 1945 – the Second World War fina...

In [5]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'sentence_transformers', '--quite'])
    from sentence_transformers import SentenceTransformer

import torch
rprint(f"GPU Available: {torch.cuda.is_available()}")

model = SentenceTransformer('BAAI/bge-base-en-v1.5', device = 'cuda' if torch.cuda.is_available() else 'cpu')

GPU Available: True

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
test_texts = catalog['text_for_embedding'].head(200).tolist()
test_embeddings = model.encode(
    test_texts,
    batch_size = 32,
    show_progress_bar = True,
    normalize_embeddings = True
)

rprint(f"Test Embedding shape: {test_embeddings.shape}")

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Test Embedding shape: (200, 768)

In [8]:
EMBEDDING_DIR = '/kaggle/working/processed/embeddings_chunks'
os.makedirs(EMBEDDING_DIR, exist_ok = True)

CHUNK_SIZE = 5000
texts = catalog['text_for_embedding'].tolist()
content_ids = catalog['content_id'].tolist()
total = len(texts)
num_chunks = (total + CHUNK_SIZE - 1) // CHUNK_SIZE

rprint(f"Total titles: {total} | Chunks: {num_chunks}")

for chunk_idx in range(num_chunks):
    chunk_path = os.path.join(EMBEDDING_DIR, f"chunk_{chunk_idx}.npy")
    if os.path.exists(chunk_path):
        rprint(f"Chunk {chunk_idx} alrady done, skipping")
        continue

    start = chunk_idx * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, total)
    chunk_texts = texts[start : end]

    chunk_embeddings = model.encode(
        chunk_texts,
        batch_size = 64,
        show_progress_bar = True,
        normalize_embeddings = True
    )
    np.save(chunk_path, chunk_embeddings)
    rprint(f"Chunk {chunk_idx} saved: rows {start}-{end}")

rprint("All chunks done.")

Total titles: 38984 | Chunks: 8

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Chunk 0 saved: rows 0-5000

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Chunk 1 saved: rows 5000-10000

...

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Chunk 6 saved: rows 30000-35000

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Chunk 7 saved: rows 35000-38984

All chunks done.

In [9]:
all_embeddings = []

for chunk_idx in range(num_chunks):
    chunk_path = os.path.join(EMBEDDING_DIR, f'chunk_{chunk_idx}.npy')
    all_embeddings.append(np.load(chunk_path))

final_embeddings = np.vstack(all_embeddings)
rprint(f"Final embedding matrix shape: {final_embeddings.shape}")


assert final_embeddings.shape[0] == len(catalog), "Row Count Mismatched!"

np.save('/kaggle/working/processed/overview_embeddings_v1.npy', final_embeddings)

pd.DataFrame({'content_id': content_ids}).to_csv(
    '/kaggle/working/processed/embedding_content_id_order.csv', index = False
)

rprint("Saved embeddings + content_id order mapping.")

Final embedding matrix shape: (38984, 768)

Saved embeddings + content_id order mapping.

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

query_idx = catalog[catalog['title'] == 'Inception'].index[0]
rprint(f"Query: {catalog.loc[query_idx, 'title']}, '-' {catalog.loc[query_idx, 'genres']}")

query_vec = final_embeddings[query_idx].reshape(1, -1)
similarities = cosine_similarity(query_vec, final_embeddings)[0]

top_indices = np.argsort(similarities)[::-1][1:6]

for idx in top_indices:
    rprint(f"{similarities[idx]:.3f} | {catalog.loc[idx, 'title']} | {catalog.loc[idx, 'genres']}")

Query: Inception, '-' ['Adventure' 'Science Fiction' 'Thriller']

0.690 | The Invisible Man | ['Action' 'Comedy' 'Science Fiction']

0.689 | Paranoia | ['Drama' 'Thriller']

0.681 | Parker | ['Action' 'Crime' 'Thriller']

0.673 | Sneakers | ['Comedy' 'Crime' 'Drama']

0.660 | Identity Thief | ['Comedy' 'Crime' 'Drama']

In [11]:
inception_matches = catalog[catalog['title'] == 'Inception']

rprint("Total matches for 'Inception':", len(inception_matches))
rprint(inception_matches[['content_id', 'title', 'year', 'genres']])

Total matches for 'Inception': 1

content_id      title    year                                  genres
2  imdb_tt1375666  Inception  2010.0  [Adventure, Science Fiction, Thriller]

In [12]:
rprint(repr(catalog.loc[query_idx, 'text_for_embedding']))

'Cobb, a skilled thief who commits corporate espionage by infiltrating the subconscious of his targets is offered a
chance to regain his old life as payment for a task considered to be impossible: "inception", the implantation of 
another person\'s idea into a target\'s subconscious.'

In [13]:
rprint("catalog index is default RangeIndex:", (catalog.index == pd.RangeIndex(len(catalog))).all())
rprint("query_idx used:", query_idx)
rprint("Row at that position in catalog:", catalog.iloc[query_idx][['title', 'content_id']])

catalog index is default RangeIndex: True

query_idx used: 2

Row at that position in catalog: title              Inception
content_id    imdb_tt1375666
Name: 2, dtype: object

In [14]:
candidate_titles = [
    'Inception', 'The Matrix', 'Shutter Island', 'Total Recall',
    'Source Code', 'Vanilla Sky', 'Minority Report', 'Memento', 'The Prestige',
    'Sneakers', 'Parker', 'Identity Thief'
]

test_rows = catalog[catalog['title'].isin(candidate_titles)].drop_duplicates('title')
rprint(test_rows[['title', 'genres']])

def build_augmented_text(row):
    genres_str = ', '.join(row['genres']) if len(row['genres']) > 0 else ''
    overview = row['overview'] if pd.notna(row['overview']) and row['overview'].strip() else row['title']
    return f"{row['title']}. Genres: {genres_str}. {overview}"

test_rows = test_rows.copy()
test_rows['augmented_text'] = test_rows.apply(build_augmented_text, axis = 1)

test_embeddings_aug = model.encode(test_rows['augmented_text'].tolist(), normalize_embeddings = True)

query_pos = test_rows.reset_index(drop = True)
query_i = query_pos[query_pos['title'] == 'Inception'].index[0]
sims = cosine_similarity(test_embeddings_aug[query_i].reshape(1,-1), test_embeddings_aug)[0]

for i in np.argsort(sims)[::-1]:
    rprint(f"{sims[i]:.3f} | {query_pos.loc[i, 'title']}")

title                                  genres
2           Inception  [Adventure, Science Fiction, Thriller]
7          The Matrix               [Action, Science Fiction]
21     Shutter Island                     [Mystery, Thriller]
23       The Prestige       [Drama, Mystery, Science Fiction]
34            Memento                 [Drama, Music, Mystery]
264   Minority Report                [Action, Crime, Mystery]
285       Source Code                [Action, Drama, Mystery]
559      Total Recall    [Action, Adventure, Science Fiction]
760       Vanilla Sky             [Fantasy, Mystery, Romance]
1833   Identity Thief                  [Comedy, Crime, Drama]
2010           Parker               [Action, Crime, Thriller]
3694         Sneakers                  [Comedy, Crime, Drama]

1.000 | Inception

0.678 | Parker

0.658 | Identity Thief

0.650 | The Prestige

...

0.607 | Total Recall

0.569 | Source Code

0.554 | Minority Report

0.550 | Memento

0.512 | Vanilla Sky

In [15]:
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

test_content_ids = test_rows['content_id'].tolist()
test_genre_vecs = genre_df.loc[test_rows.index].values

query_pos_idx = test_rows.reset_index(drop = True)
query_i = list(test_rows['title']).index('Inception')

genre_sims = cos_sim(test_genre_vecs[query_i].reshape(1,-1), test_genre_vecs)[0]

for i in np.argsort(genre_sims)[::-1]:
    rprint(f"{genre_sims[i]:.3f} | {test_rows.iloc[i]['title']} | {test_rows.iloc[i]['genres']}")

1.000 | Inception | ['Adventure' 'Science Fiction' 'Thriller']

0.667 | Total Recall | ['Action' 'Adventure' 'Science Fiction']

0.408 | The Matrix | ['Action' 'Science Fiction']

0.408 | Shutter Island | ['Mystery' 'Thriller']

0.333 | The Prestige | ['Drama' 'Mystery' 'Science Fiction']

...

0.000 | Identity Thief | ['Comedy' 'Crime' 'Drama']

0.000 | Vanilla Sky | ['Fantasy' 'Mystery' 'Romance']

0.000 | Memento | ['Drama' 'Music' 'Mystery']

0.000 | Source Code | ['Action' 'Drama' 'Mystery']

0.000 | Minority Report | ['Action' 'Crime' 'Mystery']

In [16]:
api.upload_file(
    path_or_fileobj = '/kaggle/working/processed/overview_embeddings_v1.npy',
    path_in_repo = 'overview_embeddings_v1.npy',
    repo_id = 'Subhadip007/UERP_Dataset',
    repo_type = 'dataset',
)

api.upload_file(
    path_or_fileobj = '/kaggle/working/processed/embedding_content_id_order.csv',
    path_in_repo = 'embedding_content_id_order.csv',
    repo_id = 'Subhadip007/UERP_Dataset',
    repo_type = 'dataset',
)

rprint("Embeddings pushed!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Embeddings pushed!

### **Step 3: Numerical Features**

In [17]:
catalog['year_capped'] = catalog['year'].clip(lower = 1920, upper = 2026)
catalog['year_filled'] = catalog['year_capped'].fillna(catalog['year_capped'].median())
catalog['year_normalized'] = (catalog['year_filled'] - 1920) / (2026 - 1920)

catalog['runtime_capped'] = catalog['runtime_minutes'].clip(upper = 240)
catalog['runtime_filled'] = catalog['runtime_capped'].fillna(catalog['runtime_capped'].median())
catalog['runtime_normalized'] = catalog['runtime_filled'] / 240

catalog['rating_filled'] = catalog['rating_normalized'].fillna(catalog['rating_normalized'].mean())
catalog['rating_scaled'] = catalog['rating_filled'] / 10.0

rprint(catalog[['year', 'year_normalized', 'runtime_minutes', 'runtime_normalized', 'rating_normalized', 'rating_scaled']].describe())

year  year_normalized  runtime_minutes  runtime_normalized  \
count  38823.000000     38984.000000     33110.000000        38984.000000   
mean    2005.369807         0.806279        86.795107            0.363539   
std       20.440150         0.189427        44.956586            0.153221   
min     1874.000000         0.000000         1.000000            0.004167   
25%     1999.000000         0.745283        60.000000            0.250000   
50%     2012.000000         0.867925        93.000000            0.387500   
75%     2019.000000         0.933962       110.000000            0.441667   
max     2026.000000         1.000000      1611.000000            1.000000   

       rating_normalized  rating_scaled  
count       38901.000000   38984.000000  
mean            6.755078       0.675508  
std             1.074245       0.107310  
min             1.000000       0.100000  
25%             6.200000       0.620000  
50%             6.900000       0.690000  
75%             7.500000       0.750000  
max             9.600000       0.960000

### **Step 4: Popilarity Normalization**

In [18]:
catalog['popularity_percentile'] = catalog.groupby('is_anime')['popularity_signal'].rank(pct = True)

rprint(catalog.groupby('is_anime')['popularity_percentile'].describe())
rprint()

rprint(catalog[catalog['popularity_percentile'].between(0.95, 0.96)][['title', 'is_anime', 'popularity_signal', 'popularity_percentile']].head(6))

count      mean       std       min       25%       50%       75%  \
is_anime                                                                        
False     34093.0  0.500015  0.288679  0.000073  0.250022  0.500029  0.750007   
True       4891.0  0.500102  0.288705  0.000204  0.250153  0.500102  0.750051   

          max  
is_anime       
False     1.0  
True      1.0

title  is_anime  popularity_signal  popularity_percentile
1231   Finding Neverland     False             217667               0.959992
1232  Dazed and Confused     False             217521               0.959962
1233        Run Lola Run     False             217367               0.959933
1234       Mortal Kombat     False             217235               0.959904
1235         After Earth     False             217197               0.959874
1236         City Lights     False             217181               0.959845

### **Step 5: Final Feature Metrix**

#### **Step 1: Merging Genre Metrix to Catalog**

In [20]:
catalog = pd.concat([catalog.reset_index(drop = True), genre_df.reset_index(drop = True)], axis = 1)

rprint(catalog.shape)

(38984, 89)

#### **Step 2: Creating Structure Feature Matrix**

In [21]:
genre_cols = [c for c in catalog.columns if c.startswith('genre_')]
numeric_cols = ['year_normalized', 'runtime_normalized', 'rating_scaled', 'popularity_percentile']

structured_features = catalog[genre_cols + numeric_cols].fillna(0).values.astype(np.float32)

rprint("Structured feature matrix shape:", structured_features.shape)   

np.save('/kaggle/working/processed/structured_features_v1.npy', structured_features)

Structured feature matrix shape:
(38984, 132)

#### **Step 3: Clean Up**

In [23]:
# drop_cols = ['overview_filled', 'text_for_embedding', 'year_capped', 'runtime_capped']
# catalog_final = catalog.drop(columns = [c for c in drop_cols if c in catalog.columns])

# catalog_final.to_parquet('/kaggle/working/processed/catalog_with_features_v1.parquet', index = False)

# rprint("Final feature catalog shape:", catalog_final.shape)
# rprint(catalog_final.columns.tolist())

### **&uuarr; some error occurred so I just repeated it from fresh**

In [24]:
# ============================================
# FRESH RELOAD — v4 
# ============================================

catalog_path_v4 = hf_hub_download(
    repo_id = "Subhadip007/UERP_Dataset",
    filename = "unified_catalog_v4.parquet",
    repo_type = "dataset",
    token = HF_TOKEN,
)
catalog = pd.read_parquet(catalog_path_v4)
rprint("Fresh reload:", catalog.shape)   # expect (38984, 14)

# ============================================
# Genre multi-hot encoding
# ============================================

from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(catalog['genres'])
genre_df = pd.DataFrame(genre_matrix, columns=[f"genre_{g}" for g in mlb.classes_], index = catalog.index)

# ============================================
# Numeric features
# ============================================

catalog['year_capped'] = catalog['year'].clip(lower = 1920, upper = 2026)
catalog['year_filled'] = catalog['year_capped'].fillna(catalog['year_capped'].median())
catalog['year_normalized'] = (catalog['year_filled'] - 1920) / (2026 - 1920)

catalog['runtime_capped'] = catalog['runtime_minutes'].clip(upper = 240)
catalog['runtime_filled'] = catalog['runtime_capped'].fillna(catalog['runtime_capped'].median())
catalog['runtime_normalized'] = catalog['runtime_filled'] / 240

catalog['rating_filled'] = catalog['rating_normalized'].fillna(catalog['rating_normalized'].mean())
catalog['rating_scaled'] = catalog['rating_filled'] / 10.0

catalog['popularity_percentile'] = catalog.groupby('is_anime')['popularity_signal'].rank(pct = True)

# ============================================
# Merge genre - One time Only
# ============================================

catalog = pd.concat([catalog.reset_index(drop = True), genre_df.reset_index(drop=True)], axis = 1)
rprint("After single genre merge:", catalog.shape)   # expect (38984, 14+7+32 = 53)

# ============================================
# Verify no duplicate columns
# ============================================

rprint("Any duplicate columns?", catalog.columns.duplicated().sum())

Fresh reload:
(38984, 14)

After single genre merge:
(38984, 55)

Any duplicate columns? 0

In [25]:
# ============================================
# Structured feature matrix (genre + numeric)
# ============================================

genre_cols = [c for c in catalog.columns if c.startswith('genre_')]
numeric_cols = ['year_normalized', 'runtime_normalized', 'rating_scaled', 'popularity_percentile']

rprint("Genre cols count:", len(genre_cols))       # expect 32
rprint("Numeric cols count:", len(numeric_cols))   # expect 4

structured_features = catalog[genre_cols + numeric_cols].fillna(0).values.astype(np.float32)
rprint("Structured feature matrix shape:", structured_features.shape)   # expect (38984, 36)

np.save('/kaggle/working/processed/structured_features_v1.npy', structured_features)

# ============================================
# Final catalog - clean helper column
# ============================================
drop_cols = ['year_capped', 'runtime_capped']
catalog_final = catalog.drop(columns = [c for c in drop_cols if c in catalog.columns])

rprint("Any duplicate columns in final?", catalog_final.columns.duplicated().sum())
rprint("Final feature catalog shape:", catalog_final.shape)   # expect (38984, 53)

catalog_final.to_parquet('/kaggle/working/processed/catalog_with_features_v1.parquet', index = False)

Genre cols count: 32

Numeric cols count: 4

Structured feature matrix shape:
(38984, 36)

Any duplicate columns in final? 0

Final feature catalog shape:
(38984, 53)

In [26]:
api.upload_file(
    path_or_fileobj = '/kaggle/working/processed/catalog_with_features_v1.parquet',
    path_in_repo = 'catalog_with_features_v1.parquet',
    repo_id = 'Subhadip007/UERP_Dataset',
    repo_type = 'dataset',
)

api.upload_file(
    path_or_fileobj = '/kaggle/working/processed/structured_features_v1.npy',
    path_in_repo = 'structured_features_v1.npy',
    repo_id = 'Subhadip007/UERP_Dataset',
    repo_type = 'dataset',
)

rprint("Phase 3 artifacts pushed!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Phase 3 artifacts pushed!